In [1]:
import av
import torch
import numpy as np
from transformers import AutoProcessor, LlavaNextVideoForConditionalGeneration
import cv2

In [2]:
# Define model ID
model_id = "llava-hf/LLaVA-NeXT-Video-7B-hf"

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
# Load the processor and model
model = LlavaNextVideoForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    low_cpu_mem_usage=True,
    load_in_4bit=True,
    # attn_implementation="flash_attention_2",
    )

config.json: 0.00B [00:00, ?B/s]

c:\Users\bigda\anaconda3\envs\gpu-env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bigda\.cache\huggingface\hub\models--llava-hf--LLaVA-NeXT-Video-7B-hf. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future ve

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.18G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [5]:
processor = AutoProcessor.from_pretrained(model_id)

processor_config.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

video_preprocessor_config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

In [6]:
# Function to read video frames using OpenCV
def read_video_opencv(video_path, num_frames=8):
    '''
    Decode video frames using OpenCV.
    Args:
        video_path (str): Path to the video file.
        num_frames (int): Number of frames to decode.
    Returns:
        result (np.ndarray): Array of decoded frames of shape (num_frames, height, width, 3).
    '''
    frames = []
    index = 0
    video = cv2.VideoCapture(video_path)
    fps = int(video.get(cv2.CAP_PROP_FPS))
    total_num_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.arange(0, total_num_frames, total_num_frames / num_frames).astype(int)
    while video.isOpened():
        success, frame = video.read()
        if index in indices:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        if success:
            index += 1
        if index >= total_num_frames:
            break

    video.release()

    if not frames:
        raise ValueError("No frames were read from the video.")

    return np.stack(frames)


In [7]:
conversation = [
    {
        "role": "user",
        "content": [
            {
                "type": "text", "text": "What is abnormal action in this video?",
            },
            {
                "type": "video"
            }
        ]
    }
]

prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

In [8]:
# video_path = "D:/6. Datasets/Fall-Dataset/Cut/C00_010_0004-C1.mp4"
# video_path = "D:/6. Datasets/Fight-Dataset/C00_083_0003-C1.mp4"
video_path = "D:/6. Datasets/Fight-Dataset/fight-video-2.mp4"
clip = read_video_opencv(video_path, num_frames=8)

inputs = processor(text=prompt, videos=clip, padding=True, return_tensors="pt").to(device)

In [9]:
outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False)
print(processor.decode(outputs[0], skip_special_tokens=True))

USER: 
What is abnormal action in this video? ASSISTANT: The video shows a group of people engaging in a physical altercation, with one person lying on the ground and another person standing over them. This is an abnormal action as it is not a typical or safe way to resolve conflicts. The situation appears to be escalating and could potentially lead to serious injury.


### Run Inference on unseen abnormal videos

In [15]:
import pandas as pd
import os

In [12]:
# Read unseen videos list
df_unseen_videos = pd.read_csv('../datasets/unseen_videos_information.csv')
df_unseen_videos.head()


,video_file_name,video_type,label
0,ucaerial_actions1_Walking_0_0.mp4,walking,normal
1,ucaerial_actions1_Walking_0_1.mp4,walking,normal
2,ucaerial_actions1_Walking_0_2.mp4,walking,normal
3,ucaerial_actions1_Walking_0_3.mp4,walking,normal
4,ucaerial_actions1_Walking_12_0.mp4,walking,normal


In [20]:
df_abnormal = df_unseen_videos[df_unseen_videos['label'] == 'abnormal'].reset_index(drop=True)
df_abnormal.head()

,video_file_name,video_type,label
0,bitint_box_0002.mp4,hitting,abnormal
1,bitint_box_0015.mp4,hitting,abnormal
2,bitint_box_0021.mp4,hitting,abnormal
3,bitint_box_0022.mp4,hitting,abnormal
4,bitint_box_0038.mp4,hitting,abnormal


In [21]:
df_abnormal.shape

(211, 3)

In [24]:
# Run Inference on unseen abnormal videos and Save Results to CSV
base_path = "D:/6. Datasets/SPHAR-Dataset/videos"
video_names = []
video_types = []
predicted_descriptions = []

try:
    for index, row in df_abnormal.iterrows():
        video_name = row['video_file_name']
        video_type = row['video_type']
        video_path = os.path.join(base_path, video_type, video_name)
        print(f"Processing video {index + 1}/{len(df_abnormal)}: {video_name} of type: {video_type}")

        clip = read_video_opencv(video_path, num_frames=8)

        inputs = processor(text=prompt, videos=clip, padding=True, return_tensors="pt").to(device)

        outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False)
        predicted_description = processor.decode(outputs[0], skip_special_tokens=True)
        predicted_description = predicted_description.split("ASSISTANT:")[-1].strip()

        video_names.append(video_name)
        video_types.append(video_type)
        predicted_descriptions.append(predicted_description)
except Exception as e:
    print(f"Error processing video {video_name}: {e}")


df_results = pd.DataFrame({
    'video_file_name': video_names,
    'video_type': video_types,
    'predicted_description': predicted_descriptions
})

df_results.to_csv('../outputs/llava_next_video_7b_abnormal_videos_inference_results.csv', index=False)
print("Inference results saved to CSV.")

Processing video 1/211: bitint_box_0002.mp4 of type: hitting
Processing video 2/211: bitint_box_0015.mp4 of type: hitting
Processing video 3/211: bitint_box_0021.mp4 of type: hitting
Processing video 4/211: bitint_box_0022.mp4 of type: hitting
Processing video 5/211: bitint_box_0038.mp4 of type: hitting
Processing video 6/211: bitint_box_0049.mp4 of type: hitting
Processing video 7/211: bitint_push_0001.mp4 of type: hitting
Processing video 8/211: bitint_push_0003.mp4 of type: hitting
Processing video 9/211: bitint_push_0005.mp4 of type: hitting
Processing video 10/211: bitint_push_0008.mp4 of type: hitting
Processing video 11/211: bitint_push_0009.mp4 of type: hitting
Processing video 12/211: bitint_push_0014.mp4 of type: hitting
Processing video 13/211: bitint_push_0022.mp4 of type: hitting
Processing video 14/211: bitint_push_0025.mp4 of type: hitting
Processing video 15/211: bitint_push_0043.mp4 of type: hitting
Processing video 16/211: bitint_push_0044.mp4 of type: hitting
Process